In [ ]:
import rioxarray as rxr
import xarray as xr
import numpy as np
import pandas as pd
import math
import datetime
import os
import gc

# Define the input folder path, Earth-Sun distance Excel file path, and output folder path
input_folder_path = r"C:\isro\test\R2F03JAN2025071149009500048SSANSTUC00GTDB"
earth_sun_excel_file_path=r"C:\isro\upload\TOA_help\Earth_Sun_distance.xlsx"
output_folder_path = r"C:\isro\output"
apply_dos = False  # Set to True if you want to apply Dark Object Subtraction (DOS)


# Function to convert Digital Numbers (DN) to TOA reflectance
def convert_DN_to_reflectance(input_folder, earth_sun_excel_file_path, output_folder_path, apply_dos):
    
    if not os.path.exists(input_folder_path):
        raise FileNotFoundError(f"The folder {input_folder_path} does not exist.")
    elif not os.path.exists(earth_sun_excel_file_path):
        raise FileNotFoundError(f"The Excel file does not exist at: {earth_sun_excel_file_path}")

    # Part 1 - Extracting metadata from the BAND_META.txt file
    metadata_file_path = os.path.join(input_folder, 'BAND_META.txt')

    if not os.path.exists(metadata_file_path):
        raise FileNotFoundError(f"The metadata file does not exist at: {metadata_file_path}")

    metadata = {}
    with open(metadata_file_path) as f:
        for line in f:
            key, value = line.split('=')
            metadata[key.strip()] = value.strip()

    scene_id = metadata['OTSProductID']
    print(f"Processing data for scene: {scene_id}")


    # Part 2 - Extracting date, Earth-Sun distance, E_sun and L_max & L_min values from the Excel file and metadata file
    date_str = metadata['DateOfPass']  # Extract date string from the metadata
    date = datetime.datetime.strptime(date_str, '%d-%b-%Y').replace(year=2020)  # Date is in the string format 03-JAN-2025, we need to convert it to a datetime object

    excel_file = pd.read_excel(earth_sun_excel_file_path, sheet_name="Earth_sun_distance", header=1)  # Read the 'Earth_sun_distance' sheet of the excel file  # Columns are in the second row (header=1)

    match = excel_file.where(excel_file == date).stack()  # `== target_date` checks for exact match  #`.stack()` converts and returns the DataFrame into a Series for easier searching
    if not match.empty:
        row_idx, col_label = match.index[0]  # - `.index` gives the indices of the matches in string format
        col_idx = excel_file.columns.get_loc(col_label)  # Get the column index of the matched date string
        earth_sun_distance = excel_file.iloc[row_idx, col_idx + 2]  # Extract earth-sun distance in Astronomical Units (AU)
    else:
        print(f"Target date: {date} not found in the Excel file.")

    E_sun = {
        'RS2': {
            'B2': 185.33,
            'B3': 157.766,
            'B4': 111.359
        },
        'RS2A': {
            'B2': 185.347,
            'B3': 158.262,
            'B4': 110.81
        }
    }

    if metadata['SatID'] == 'IRS-R2':
        b2_esun = E_sun['RS2']['B2']  # Extracting E_sun value for Band 2 (Green) from the E_sun dictionary
        b3_esun = E_sun['RS2']['B3']  # Extracting E_sun value for Band 3 (Red) from the E_sun dictionary
        b4_esun = E_sun['RS2']['B4']  # Extracting E_sun value for Band 4 (NIR) from the E_sun dictionary
    elif metadata['SatID'] == 'IRS-R2A':
        b2_esun = E_sun['RS2A']['B2']  # Extracting E_sun value for Band 2 (Green) from the E_sun dictionary
        b3_esun = E_sun['RS2A']['B3']  # Extracting E_sun value for Band 3 (Red) from the E_sun dictionary
        b4_esun = E_sun['RS2A']['B4']  # Extracting E_sun value for Band 4 (NIR) from the E_sun dictionary
    else:
        raise ValueError(f"Unknown satellite ID: {metadata['SatID']}")
    
    B2_Lmax = float(metadata['B2_Lmax'])  # Extracting L_max value for Band 2 (Green) from the metadata
    B2_Lmin = float(metadata['B2_Lmin'])  # Extracting L_min value for Band 2 (Green) from the metadata
    B3_Lmax = float(metadata['B3_Lmax'])  # Extracting L_max value for Band 3 (Red) from the metadata
    B3_Lmin = float(metadata['B3_Lmin'])  # Extracting L_min value for Band 3 (Red) from the metadata
    B4_Lmax = float(metadata['B4_Lmax'])  # Extracting L_max value for Band 4 (NIR) from the metadata
    B4_Lmin = float(metadata['B4_Lmin'])  # Extracting L_min value for Band 4 (NIR) from the metadata
    sun_elevation_angle = math.radians(float(metadata['SunElevationAtCenter']))  # Extracting Sun elevation angle at the center of the scene from the metadata and converting it to radians


    # Part 3 - Calculating TOA reflectance for each band and saving the output as a GeoTIFF file
    b2_dn_val = rxr.open_rasterio(os.path.join(input_folder, 'BAND2.tif'))  # Open the raster file for Band 2 (Green)
    b3_dn_val = rxr.open_rasterio(os.path.join(input_folder, 'BAND3.tif'))  # Open the raster file for Band 3 (Red)
    b4_dn_val = rxr.open_rasterio(os.path.join(input_folder, 'BAND4.tif'))  # Open the raster file for Band 4 (NIR)
    
    if b2_dn_val is None:
        raise FileNotFoundError("Cannot read BAND2.tif. File is empty or corrupted or missing.")
    elif b3_dn_val is None:
        raise FileNotFoundError("Cannot read BAND3.tif. File is empty or corrupted or missing.")
    elif b4_dn_val is None:
        raise FileNotFoundError("Cannot read BAND4.tif. File is empty or corrupted or missing.")
    
    b2_dn = b2_dn_val.where(b2_dn_val != 0, float('nan'))  # Replace 0 values with NaN to avoid division by zero errors
    b3_dn = b3_dn_val.where(b3_dn_val != 0, float('nan'))  # Replace 0 values with NaN to avoid division by zero errors
    b4_dn = b4_dn_val.where(b4_dn_val != 0, float('nan'))  # Replace 0 values with NaN to avoid division by zero errors

    del b2_dn_val, b3_dn_val, b4_dn_val  # Free up memory by deleting the original DN value arrays
    gc.collect()  # Force garbage collection to free up memory


    b2_rad = (b2_dn * (B2_Lmax - B2_Lmin)) / 1024  # Convert digital numbers to radiance for Band 2 (Green)
    b3_rad = (b3_dn * (B3_Lmax - B3_Lmin)) / 1024  # Convert digital numbers to radiance for Band 3 (Red)
    b4_rad = (b4_dn * (B4_Lmax - B4_Lmin)) / 1024  # Convert digital numbers to radiance for Band 4 (NIR)

    del b2_dn, b3_dn, b4_dn  # Free up memory by deleting the original DN arrays
    gc.collect()  # Force garbage collection to free up memory
    

    b2_ref = (math.pi*b2_rad*earth_sun_distance*earth_sun_distance)/(b2_esun*math.sin(sun_elevation_angle))  # Calculate TOA reflectance for Band 2 (Green)
    b3_ref = (math.pi*b3_rad*earth_sun_distance*earth_sun_distance)/(b3_esun*math.sin(sun_elevation_angle))  # Calculate TOA reflectance for Band 3 (Red)
    b4_ref = (math.pi*b4_rad*earth_sun_distance*earth_sun_distance)/(b4_esun*math.sin(sun_elevation_angle))  # Calculate TOA reflectance for Band 4 (NIR)

    del b2_rad, b3_rad, b4_rad  # Free up memory by deleting the original radiance arrays
    gc.collect()  # Force garbage collection to free up memory


    # Part 4 - Generating Cloud Optimized GeoTIFF (COG) files as output
    if not apply_dos:
        print("apply_dos set to False. Dark Object Subtraction (DOS) is not applied. Using original reflectance values.")
        scene_ref = xr.concat([b2_ref, b3_ref, b4_ref], dim='band').assign_coords(band=['BAND2', 'BAND3', 'BAND4'])
        scene_ref.name = scene_id

        del b2_ref, b3_ref, b4_ref  # Free up memory by deleting the original reflectance arrays
        gc.collect()  # Force garbage collection to free up memory
    
    else:
        print("apply_dos set to True. Applying Dark Object Subtraction (DOS) to the reflectance values.")

        # Wavelengths (µm) for Resourcesat - 2/2A LISS-IV (B2=Green, B3=Red, B4=NIR)
        wavelengths = {
            'BAND2': 0.555,  # Green
            'BAND3': 0.650, # Red
            'BAND4': 0.815  # NIR (not used in your reference, but included for completeness)
        }

        # Find dark object value for reference
        dark = None
        red_band = b3_ref.values  # Extract the red band (BAND3) as reference
        change = np.nanmax(red_band) / 250  # Threshold for change detection
        for pixel in np.unique(red_band[~np.isnan(red_band)])[:100]:  # Skip NaN values
            if pixel == 0:
                continue
            if dark is not None and (pixel - dark) < change:
                break
            else:
                dark = pixel

        if dark is None:
            raise ValueError("No valid dark object found in the red band_val.")

        # Dynamically adjust model based on median of green and NIR bands
        green_median = np.nanmedian(b2_ref.values)
        nir_median = np.nanmedian(b4_ref.values)
        if green_median > 0.15:  # Heavy haze
            model = -1
        elif nir_median < 0.02:  # Clear
            model = -4
        else:  # Moderate haze
            model = -2
        
        # Dynamically adjust mult based on dark value
        if dark < 0.01:  # Very dark scene (low haze)
            multiplier =  0.8
        elif dark > 0.05:  # Bright scene (high haze)
            multiplier =  1.2
        else:
            multiplier =  1.0

        # Calculate haze reduction factors (same formula as reference)
        b2_haze = dark * ((wavelengths['BAND2'] ** model) / (wavelengths['BAND3'] ** model)) * multiplier  # Green
        b3_haze = dark * ((wavelengths['BAND3'] ** model) / (wavelengths['BAND3'] ** model)) * multiplier  # Red
        b4_haze = dark * ((wavelengths['BAND4'] ** model) / (wavelengths['BAND3'] ** model)) * multiplier  # NIR 

        # Subtract haze and clip negative values
        b2_ref_corrected = (b2_ref - b2_haze).clip(min=0, max=1.0)
        b3_ref_corrected = (b3_ref - b3_haze).clip(min=0, max=1.0)
        b4_ref_corrected = (b4_ref - b4_haze).clip(min=0, max=1.0)

        del b2_ref, b3_ref, b4_ref  # Free up memory by deleting the original DN arrays
        gc.collect()  # Force garbage collection to free up memory


        scene_ref = xr.concat([b2_ref_corrected, b3_ref_corrected, b4_ref_corrected], dim='band').assign_coords(band=['BAND2', 'BAND3', 'BAND4'])
        scene_ref.name = scene_id    

        del b2_ref_corrected, b3_ref_corrb2_ref_corrected, b4_ref_corrb2_ref_corrected  # Free up memory by deleting the original DN arrays
        gc.collect()  # Force garbage collection to free up memory


    os.makedirs(output_folder_path, exist_ok=True) # Create the output folder if it does not exist
    output_ds = scene_ref.to_dataset('band')
    output_file_path = os.path.join(output_folder_path, f'{scene_id}.tif')
    output_options = {
        'num_threads': 4,  # Use multiple threads for faster processing
        'driver': 'COG',  # Use COG driver for Cloud Optimized GeoTIFF
        'dtype': 'float32',  # Set data type to float32 for reflectance values
        'predictor': 3,  # Better compression for floats
        'quality': 1,  # Set quality to 1 for best quality
        'blockxsize': 256,  # Set block size for tiled GeoTIFF
        'blockysize': 256,  # Set block size for tiled GeoTIFF
        'nodata': 0,  # Set nodata value to 0
        'BIGTIFF': 'IF_NEEDED',  # Use BigTIFF format for large files
        'compress': 'deflate',  # Use deflate compression for better compression ratio
        'overviews': 'auto',       # Build pyramids for cloud
        'tiled': True,  # set True if you want to create a tiled GeoTIFF
        'windowed': True # set True if you run out of RAM
    }

    output_ds[['BAND2', 'BAND3', 'BAND4']].rio.to_raster(output_file_path, **output_options)
    print(f'Output file created successfully at: {output_file_path}')


# Call the function to convert DN to reflectance
convert_DN_to_reflectance(input_folder_path, earth_sun_excel_file_path, output_folder_path, apply_dos)


In [1]:
import rasterio

with rasterio.open(r'C:\isro\test\R2F03JAN2025071149009500048SSANSTUC00GTDB\BAND3.tif') as src:
    print(src.dtypes)  # Returns a tuple of dtypes (e.g., ('float32',))
    print(src.meta)    # Full metadata, including dtype

('uint16',)
{'driver': 'GTiff', 'dtype': 'uint16', 'nodata': None, 'width': 17967, 'height': 17410, 'count': 1, 'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 43N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.25722293287,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",75],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32643"]]'), 'transform': Affine(5.0, 0.0, 714658.943083703,
       0.0, -5.0, 3630532.5)}
